In [1]:
from Indexing import preprocess_and_store
from Retriver import Retriever

from dotenv import load_dotenv

from langchain.chat_models import init_chat_model
from langchain_groq import ChatGroq


/home/abdelrhman/Desktop/Deep Learning /venv-lc/lc_env/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
load_dotenv()

True

In [3]:
llm = ChatGroq(
    model="llama-3.1-8b-instant",
    temperature=0.1
)



In [4]:
input = "If the transport services needed by an application don’t quite fit either the UDP or TCP service models—perhaps an application needs more services than those provided by UDP but does not want all of the particular functionality that comes with TCP, or may want different services than those provided by TCP—applica- tion designers can always “roll their own” protocol at the application layer. This is the approach taken in the QUIC (Quick UDP Internet Connections) protocol [Langley 2017, QUIC 2020]. Specifically, QUIC is a new application-layer pro- tocol designed from the ground up to improve the performance of transport-layer services for secure HTTP. QUIC has already been widely deployed, although is still in the process of being standardized as an Internet RFC [QUIC 2020]. Google has deployed QUIC on many of its public-facing Web servers, in its mobile video streaming YouTube app, in its Chrome browser, and in Android’s Google Search app. With more than 7% of Internet traffic today now being QUIC [Langley 2017], we’ll want to take a closer look. Our study of QUIC will also serve as a nice culmi- nation of our study of the transport layer, as QUIC uses many of the approaches for reliable data transfer, congestion control, and connection management that we’ve studied in this chapter."

prompt = f"""
You are optimizing a search query for retrieving relevant technical context from a book.

Your goal is to MAXIMIZE retrieval recall.

Instructions:
- Extract ONLY the most important technical terms and concepts
- Preserve exact wording from the paragraph (no paraphrasing)
- Include protocol names, algorithms, standards, and key mechanisms
- Prefer specific terms over general descriptions
- Ignore filler words and explanations
- Output a flat list of keywords (no sentences)

Constraints:
- Max 10 terms
- Separate terms with commas
- No duplicates

Paragraph:
{input}

Search query:
"""

response = llm.invoke(prompt)
print(response.content)

QUIC, UDP, TCP, application-layer protocol, transport-layer services, HTTP, Internet RFC, congestion control, connection management, reliable data transfer.


In [5]:
retriver = Retriever()
retriver.load_vector_store("vector_store")

content, metadata = retriver.retrieve(query=response.content, k=10)


/home/abdelrhman/Desktop/Deep Learning /Retriver.py:10: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the `langchain-huggingface package and should be used instead. To use it run `pip install -U `langchain-huggingface` and import as `from `langchain_huggingface import HuggingFaceEmbeddings``.
  self.embeddings = HuggingFaceEmbeddings(
Loading weights: 100%|██████████| 103/103 [00:00<00:00, 11514.52it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [6]:
print("Retrieved content:")
for i, (doc, meta) in enumerate(zip(content, metadata)):
    print(f"Document {i+1}:")
    print(f"Content: {doc}")
    print(f"Metadata: {meta}")
    print("-" * 40)

Retrieved content:
Document 1:
Content: Figure 2.5 indicates the transport protocols used by some popular Internet appli- cations. We see that e-mail, remote terminal access, the Web, and file transfer all use TCP. These applications have chosen TCP primarily because TCP provides reliable data transfer, guaranteeing that all data will eventually get to its destination. Because Internet telephony applications (such as Skype) can often tolerate some loss but require a minimal rate to be effective, developers of Internet telephony applications

Application Application-Layer Protocol Underlying Transport Protocol

Electronic mail SMTP [RFC 5321] TCP Remote terminal access Telnet [RFC 854] TCP Web HTTP 1.1 [RFC 7230] TCP File transfer FTP [RFC 959] TCP Streaming multimedia HTTP (e.g., YouTube), DASH TCP Internet telephony SIP [RFC 3261], RTP [RFC 3550], or proprietary UDP or TCP

Figure 2.5 ♦ Popular Internet applications, their application-layer

protocols, and their underlying transport p

In [32]:
def score_chunk(paragraph, chunk):
    prompt = f"""
You are a STRICT evaluator for retrieval-augmented translation.

Return ONLY a number (1-10).

Your job is to decide if the context is ACTUALLY USEFUL for translating the paragraph.

A chunk is USEFUL only if it:
- Explains specific terms in the paragraph
- Provides missing technical meaning
- Clarifies concepts that affect translation

A chunk is NOT useful if it:
- Is only loosely related to the topic
- Repeats general knowledge
- Mentions similar concepts but not this exact content

Scoring rules (be harsh):
1-2 = completely useless (ignore)
3-4 = weakly related, not helpful
5-6 = somewhat related but not needed
7-8 = helpful
9-10 = critical for correct translation

IMPORTANT:
- Most chunks should score between 1–6
- Only give 9–10 if the chunk is ESSENTIAL
- Avoid middle scores unless truly uncertain

Paragraph:
{paragraph}

Context:
{chunk}


Score (1-10):
"""
    return int(llm.invoke(prompt).content.strip())

In [33]:
for i, (doc, meta) in enumerate(zip(content, metadata)):
    score = score_chunk(input, doc)
    print(f"Chunk {i+1} score: {score}")


Chunk 1 score: 8
Chunk 2 score: 8
Chunk 3 score: 8
Chunk 4 score: 9
Chunk 5 score: 9
Chunk 6 score: 8
Chunk 7 score: 8
Chunk 8 score: 8
Chunk 9 score: 8
Chunk 10 score: 5
